# Modegen — подключение Kaggle к сайту

1. Создайте **новый приватный notebook** и импортируйте этот файл.
2. Включите **GPU T4** (T4 ×2 тоже подходит) и **Internet On**. Input добавлять не нужно.
3. Нажмите **Run All в редакторе**, не Save & Run All. Последняя ячейка должна оставаться работающей, пока вы пользуетесь сайтом.
4. Дождитесь строк «Адрес подключения» и «Ключ подключения». На https://modegen.ru/ai-platform/ откройте студию и вставьте их в «Подключение Kaggle».
5. Загрузите фото человека или выберите модель; загрузите разложенную футболку/рубашку/лонгслив; нажмите «Создать кадр».
6. После теста остановите сессию Kaggle, чтобы не расходовать GPU-квоту.

Это **последний завершённый тест FASHN v1.1**: закреплённые исходники и веса,
FP32, seed42, 30 шагов, CFG1.5, segmentation_free=True, верх, flat-lay,
чанк attention512, препроцессингCPU. Нового обучения и LoRA нет.
Настройки теста v1.2 с маской сюда не перенесены: его GPU-результат не получен.
Результаты v1.1 пока теряли принт; подключение к сайту не улучшает модель.

Подключение временное: HTTPS-туннель Gradio ведёт в эту GPU-сессию. Любой запрос
с фото и любой результат требуют ключа; ключ не вшит в сайт и не передаётся в URL.
Фото идут только в вашу сессию через туннель, сервер не пишет их в файлы.
**Не публикуйте вывод notebook с ключом.** Новый запуск создаёт новый ключ.
Лимит пилота — 6 попыток, 30 минут простоя / 2 часа работы; автоповторов нет.
Остановка веб-сервера не заменяет остановку самой GPU-сессии Kaggle.

Сервер и браузерный протокол проверены локально на CPU с тестовой заглушкой.
Реальная модель проверялась отдельно в v1.1. Новый полный путь сайт → Kaggle →
изображение требует первого ручного GPU-запуска этого notebook.
Никакой оплаты не подключается. Не запускайте параллельно с другим GPU-notebook.


In [ ]:
from pathlib import Path
import subprocess, sys, json
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Сначала включите GPU в настройках Kaggle.')
ROOT=Path('/kaggle/working/modegen_web')
ROOT.mkdir(parents=True,exist_ok=True)
SCRIPTS = {'engine': '"""FASHN engine copied from the completed Modegen v1.1 GPU probe. No training or private assets."""\n\nimport os\nfrom pathlib import Path\n\nTRANSFORMERS_BACKENDS = {\n    \'USE_TORCH\': \'1\', \'USE_TF\': \'0\', \'USE_FLAX\': \'0\',\n    \'FORCE_TF_AVAILABLE\': \'0\',\n}\n\nos.environ.update(TRANSFORMERS_BACKENDS)\n\nSOURCES = {\n    \'fashn-vton-1.5\': (\'https://github.com/fashn-AI/fashn-vton-1.5.git\',\n                     \'7c0f10af3f91ad4048fe9729c470a13ef905d25a\'),\n    \'fashn-human-parser\': (\'https://github.com/fashn-AI/fashn-human-parser.git\',\n                         \'f2771f2fb8655349e87e2869bbde7ace0bd06f2c\'),\n}\n\nWEIGHTS = [\n    (\'fashn-ai/fashn-vton-1.5\', \'7720683168567eb5a2a4c67f15116c6e29c83ded\',\n     \'\', [\'model.safetensors\']),\n    (\'fashn-ai/DWPose\', \'548b5df25b84d9f4aac0611dfa1c2a7a12f15571\',\n     \'dwpose\', [\'yolox_l.onnx\', \'dw-ll_ucoco_384.onnx\']),\n    (\'fashn-ai/fashn-human-parser\', \'1f80c34dbab321c5730dda5c3fea279fd3e97498\',\n     \'human_parser\', [\'config.json\', \'model.safetensors\']),\n]\n\ndef memory_estimate(query_chunk=512):\n    """Exact storage/score sizes, NOT a measured or guaranteed total VRAM peak.\n\n    Source: pinned TryOnModel defaults; independently matches HF metadata.\n    971,813,808 parameters + 432 unused checkpoint-compatible buffer values.\n    CFG doubles batch: 2 x 10 heads x 6912 tokens (two 72x48 images).\n    """\n    h = 1280\n    linear = lambda a, b: a * b + b\n    single = linear(h, 7*h) + linear(5*h, h) + 256 + linear(h, 3*h)\n    stream = (linear(h, 6*h) + linear(h, 3*h) + 256 + linear(h, h)\n              + linear(h, 4*h) + linear(4*h, h))\n    count = (linear(256, h) + linear(h, h) + 4*h + 11*h*144 + 2*h\n             + 20*single + 16*stream + linear(h, 432) + linear(h, 2*h) + 432)\n    assert count == 971814240\n    tokens = 2 * (864 // 12) * (576 // 12)\n    queries = min(query_chunk, tokens) if query_chunk else tokens\n    return dict(parameters=count-432, buffer_values=432, stored_values=count,\n                fp32_model_bytes=count*4, cfg_batch=2, sequence_tokens=tokens,\n                full_score_bytes=2*10*tokens*tokens*4,\n                chunk_score_bytes=2*10*queries*tokens*4,\n                note=\'Storage and one score tensor only; other live tensors/workspaces are additional.\')\n\ndef download_weights(out):\n    from huggingface_hub import hf_hub_download\n    root = out / \'weights\'\n    for repo, revision, folder, names in WEIGHTS:\n        for name in names:\n            hf_hub_download(repo_id=repo, revision=revision, filename=name,\n                            local_dir=str(root / folder), token=False)\n    return root\n\ndef make_pipeline(weights, preprocess_device, query_chunk):\n    import torch\n    from torch.nn import functional as F\n    from safetensors.torch import load_file\n    from fashn_vton import TryOnPipeline\n    from fashn_vton.dwpose import DWposeDetector\n    from fashn_human_parser import FashnHumanParser\n    from fashn_vton import tryon_mmdit\n\n    # Each query still attends to ALL keys. No mask/CFG/precision change.\n    # Query slicing bounds math-SDPA score storage even if fused FP32 is unavailable.\n    if query_chunk:\n        def chunked_attention(q, k, v):\n            return torch.cat([\n                F.scaled_dot_product_attention(q[:, :, i:i+query_chunk], k, v)\n                for i in range(0, q.shape[2], query_chunk)\n            ], dim=2)\n        tryon_mmdit._attn_processor = chunked_attention\n    else:\n        tryon_mmdit._attn_processor = F.scaled_dot_product_attention\n\n    class ProbePipeline(TryOnPipeline):\n        def _setup_tryon_model(self):\n            self.inference_dtype = torch.float32  # Explicit even on a newer GPU.\n            self.tryon_model = tryon_mmdit.TryOnModel()\n            state = load_file(str(weights / \'model.safetensors\'), device=\'cpu\')\n            self.tryon_model.load_state_dict(state, strict=True)\n            del state  # Avoid staging the BF16 checkpoint on cuda:0 as well.\n            self.tryon_model.to(self.device, dtype=torch.float32).eval().requires_grad_(False)\n\n        def _setup_pose_model(self):\n            self.pose_model = DWposeDetector(\n                checkpoints_dir=str(weights / \'dwpose\'), device=preprocess_device)\n\n        def _setup_hp_model(self):\n            # model_id accepts a local from_pretrained directory; no implicit main download.\n            self.hp_model = FashnHumanParser(\n                model_id=str(weights / \'human_parser\'), device=preprocess_device)\n\n    pipe = ProbePipeline(weights_dir=str(weights), device=\'cuda:0\')\n    tensors = list(pipe.tryon_model.parameters()) + list(pipe.tryon_model.buffers())\n    if sum(t.numel() for t in tensors) != memory_estimate()[\'stored_values\']:\n        raise RuntimeError(\'Generator shape/count differs from the pinned model.\')\n    if any(t.dtype != torch.float32 or str(t.device) != \'cuda:0\' for t in tensors):\n        raise RuntimeError(\'Generator must be entirely FP32 on cuda:0.\')\n    if str(next(pipe.hp_model.model.parameters()).device) != preprocess_device:\n        raise RuntimeError(\'Human parser device mismatch.\')\n    provider = \'CPUExecutionProvider\' if preprocess_device == \'cpu\' else \'CUDAExecutionProvider\'\n    wholebody = pipe.pose_model.pose_estimation\n    for session in (wholebody.session_det, wholebody.session_pose):\n        if session.get_providers()[0] != provider:\n            raise RuntimeError(\'Pose provider fell back; rerun with CPU preprocessing.\')\n        if preprocess_device == \'cuda:1\':\n            if session.get_provider_options()[provider].get(\'device_id\') != \'1\':\n                raise RuntimeError(\'Pose CUDA provider is not on device 1.\')\n    return pipe\n', 'setup_runtime': '"""Future Kaggle setup: preserve installed Torch/CUDA, fetch code only, no weights."""\nimport argparse\nimport importlib.metadata as metadata\nimport json\nfrom pathlib import Path\nimport subprocess\nimport sys\n\nfrom engine import SOURCES, TRANSFORMERS_BACKENDS\nDEFAULT_OUT = "/kaggle/working/modegen_web"\n\nIMPORT_PREFLIGHT = \'\'\'\nimport sys\nimport torch, torchvision, onnxruntime\nfrom transformers.utils import is_torch_available, is_tf_available, is_flax_available\nassert is_torch_available(), \'PyTorch backend must be available.\'\nassert not is_tf_available() and not is_flax_available(), \'Optional backends were not disabled.\'\nfrom fashn_vton import TryOnPipeline\nfrom fashn_human_parser import FashnHumanParser\nunexpected = sorted(n for n in sys.modules if n.split(\'.\')[0] in (\'tensorflow\', \'jax\', \'flax\'))\nassert not unexpected, \'Unexpected optional backend import: \' + repr(unexpected[:5])\nprint(\'Import-only preflight passed: PyTorch only; TF/JAX/Flax not imported; no weights loaded.\')\n\'\'\'\n\n\ndef run(args):\n    subprocess.run(args, check=True)\n\n\ndef protected_runtime():\n    return {d.metadata[\'Name\']: d.version for d in metadata.distributions()\n            if d.metadata[\'Name\'].lower().replace(\'_\', \'-\').startswith((\'torch\', \'nvidia-\'))\n            or d.metadata[\'Name\'].lower() == \'triton\'}\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--output\', default=DEFAULT_OUT)\n    parser.add_argument(\'--preprocess-device\', choices=[\'cpu\', \'cuda:1\'], default=\'cpu\')\n    args = parser.parse_args()\n    out = Path(args.output)\n    out.mkdir(parents=True, exist_ok=True)\n    import torch\n    from packaging.version import Version\n    if sys.version_info < (3, 10) or Version(torch.__version__.split(\'+\')[0]) < Version(\'2.2\'):\n        raise RuntimeError(\'Requires existing Python >=3.10 and Torch >=2.2 (parser requirement). No Torch replacement.\')\n    before = protected_runtime()\n    metadata.version(\'torchvision\')  # Fail before pip if the preinstalled pair is incomplete.\n    if args.preprocess_device == \'cuda:1\':\n        if (not (torch.version.cuda or \'\').startswith(\'12.\')\n                or (torch.backends.cudnn.version() or 0) < 90000):\n            raise RuntimeError(\'ORT 1.20.1 GPU path needs CUDA 12/cuDNN 9; use CPU preprocessing.\')\n        if torch.cuda.device_count() < 2:\n            raise RuntimeError(\'cuda:1 needs two visible GPUs; use CPU preprocessing.\')\n    constraint = out / \'preserve_runtime.txt\'\n    constraint.write_text(\'\'.join(f\'{name}=={version}\\n\' for name, version in sorted(before.items())))\n    ort = \'onnxruntime\' if args.preprocess_device == \'cpu\' else \'onnxruntime-gpu\'\n    other = \'onnxruntime-gpu\' if ort == \'onnxruntime\' else \'onnxruntime\'\n    try:\n        metadata.version(other)\n    except metadata.PackageNotFoundError:\n        pass\n    else:\n        run([sys.executable, \'-m\', \'pip\', \'uninstall\', \'-y\', other])\n        # Both ORT distributions share files: reinstall the selected one after removal.\n        run([sys.executable, \'-m\', \'pip\', \'uninstall\', \'-y\', ort])\n    deps = [\'numpy>=1.26,<2\', \'transformers==4.46.3\', \'huggingface_hub>=0.23,<1\',\n            \'gradio==5.49.1\', \'safetensors>=0.4\', \'Pillow>=9\', \'einops>=0.6\', \'tqdm>=4.65\',\n            \'opencv-python==4.10.0.84\', \'matplotlib>=3.5\', f\'{ort}==1.20.1\']\n    run([sys.executable, \'-m\', \'pip\', \'install\', \'--quiet\', \'--constraint\', str(constraint),\n         \'--upgrade-strategy\', \'only-if-needed\', *deps])\n    for name, (url, revision) in SOURCES.items():\n        repo = out / \'source\' / name\n        if not repo.exists():\n            repo.parent.mkdir(parents=True, exist_ok=True)\n            run([\'git\', \'clone\', \'--no-checkout\', url, str(repo)])\n        run([\'git\', \'-C\', str(repo), \'checkout\', \'--detach\', revision])\n        actual = subprocess.check_output([\'git\', \'-C\', str(repo), \'rev-parse\', \'HEAD\'], text=True).strip()\n        if actual != revision:\n            raise RuntimeError(\'Source pin mismatch: \' + name)\n        # Explicit deps above; the upstream VTON package requests onnxruntime-gpu even on CPU.\n        run([sys.executable, \'-m\', \'pip\', \'install\', \'--quiet\', \'--no-deps\', str(repo)])\n    if protected_runtime() != before:\n        raise RuntimeError(\'Torch/CUDA package versions changed unexpectedly.\')\n    # Fresh process avoids stale imports after pip; constructors/weights are never invoked.\n    run([sys.executable, \'-c\', IMPORT_PREFLIGHT])\n    (out / \'versions.json\').write_text(json.dumps(dict(\n        python=sys.version, torch=torch.__version__, cuda=torch.version.cuda,\n        runtime_preserved=before, sources=SOURCES, preprocess_device=args.preprocess_device,\n        transformers_backends=TRANSFORMERS_BACKENDS), indent=2))\n    with (out / \'pip_freeze.txt\').open(\'w\') as stream:\n        subprocess.run([sys.executable, \'-m\', \'pip\', \'freeze\'], stdout=stream, check=True)\n    print(\'Setup complete; existing Torch/CUDA preserved. No weights downloaded.\', flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n', 'server': '"""Private, temporary website bridge. Model inference stays inside Kaggle."""\nimport base64\nimport binascii\nimport hmac\nimport io\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport secrets\nimport threading\nimport time\n\nos.environ.update(USE_TORCH=\'1\', USE_TF=\'0\', USE_FLAX=\'0\', FORCE_TF_AVAILABLE=\'0\',\n                  GRADIO_ANALYTICS_ENABLED=\'False\', HF_HUB_DISABLE_TELEMETRY=\'1\')\n\nMODEL_ID = \'modegen-fashn-v11-fp32\'\nMAX_IMAGE_BYTES = 8 * 1024 * 1024\nMAX_BODY = 24 * 1024 * 1024\nMAX_PIXELS = 16_000_000\nSETTINGS = dict(model_id=MODEL_ID, seed=42, steps=30, guidance_scale=1.5,\n                segmentation_free=True, category=\'tops\', garment_photo_type=\'flat-lay\',\n                dtype=\'float32\', attention_query_chunk=512, training_steps=0,\n                weights_revision=\'7720683168567eb5a2a4c67f15116c6e29c83ded\')\nORIGINS = {\'https://modegen.ru\', \'https://www.modegen.ru\'}\n\n\nclass AccessGuard:\n    """Check the key before Gradio parses/enqueues any payload; never use URL tokens."""\n    def __init__(self, app, key, origins=ORIGINS):\n        self.app, self.key, self.origins = app, key, set(origins)\n\n    async def __call__(self, scope, receive, send):\n        if scope[\'type\'] != \'http\':\n            return await self.app(scope, receive, send)\n        from starlette.responses import JSONResponse\n        headers = dict(scope[\'headers\'])\n        origin = headers.get(b\'origin\', b\'\').decode()\n        path = scope[\'path\']\n        async def reject(code, message):\n            await JSONResponse({\'error\': message}, status_code=code)(scope, receive, send)\n        if origin and origin not in self.origins:\n            return await reject(403, \'Origin not allowed\')\n        if scope[\'method\'] == \'OPTIONS\':\n            if not origin:\n                return await reject(403, \'Origin required\')\n            response = JSONResponse({}, headers={\n                \'Access-Control-Allow-Origin\': origin,\n                \'Access-Control-Allow-Methods\': \'GET, POST, OPTIONS\',\n                \'Access-Control-Allow-Headers\': \'Authorization, Content-Type\',\n                \'Vary\': \'Origin\'})\n            return await response(scope, receive, send)\n        # Gradio\'s launch() performs this local startup request before creating its tunnel.\n        local = (scope.get(\'client\') or (\'\',))[0] in (\'127.0.0.1\', \'::1\')\n        startup = path == \'/gradio_api/startup-events\' and local and not origin\n        if path == \'/\' and scope[\'method\'] in (\'GET\', \'HEAD\'):\n            return await JSONResponse({\'service\': \'Modegen private GPU session\'})(scope, receive, send)\n        if not startup:\n            supplied = headers.get(b\'authorization\', b\'\')\n            if not hmac.compare_digest(supplied, (\'Bearer \' + self.key).encode()):\n                return await reject(401, \'Invalid session key\')\n            if not re.fullmatch(r\'/gradio_api/call/(status|tryon)(/[a-f0-9]+)?\', path):\n                return await reject(404, \'Not found\')\n        try:\n            if int(headers.get(b\'content-length\', b\'0\')) > MAX_BODY:\n                return await reject(413, \'Request too large\')\n        except ValueError:\n            return await reject(400, \'Invalid length\')\n        # Count actual bytes as well (including chunked requests) before forwarding.\n        if scope[\'method\'] == \'POST\':\n            chunks, total = [], 0\n            while True:\n                message = await receive()\n                if message[\'type\'] == \'http.disconnect\':\n                    return\n                total += len(message.get(\'body\', b\'\'))\n                if total > MAX_BODY:\n                    return await reject(413, \'Request too large\')\n                chunks.append(message)\n                if not message.get(\'more_body\'):\n                    break\n            original_receive = receive\n            async def replay():\n                return chunks.pop(0) if chunks else await original_receive()\n            receive = replay\n        async def cors_send(message):\n            if message[\'type\'] == \'http.response.start\':\n                extra = [(b\'cache-control\', b\'no-store\')]\n                if origin:\n                    extra += [(b\'access-control-allow-origin\', origin.encode()), (b\'vary\', b\'Origin\')]\n                message[\'headers\'] = message.get(\'headers\', []) + extra\n            await send(message)\n        await self.app(scope, receive, cors_send)\n\n\ndef decode_image(value):\n    from PIL import Image, ImageOps\n    if not isinstance(value, str) or len(value) > MAX_IMAGE_BYTES * 4 // 3 + 100:\n        raise ValueError(\'Фото должно быть не больше 8 МБ.\')\n    match = re.fullmatch(r\'data:image/(png|jpeg|webp);base64,([A-Za-z0-9+/=]+)\', value)\n    if not match:\n        raise ValueError(\'Нужен JPG, PNG или WebP.\')\n    try:\n        data = base64.b64decode(match[2], validate=True)\n        if len(data) > MAX_IMAGE_BYTES:\n            raise ValueError(\'Фото больше 8 МБ.\')\n        with Image.open(io.BytesIO(data)) as image:\n            if image.format not in (\'JPEG\', \'PNG\', \'WEBP\'):\n                raise ValueError(\'Неподдерживаемый формат фото.\')\n            if min(image.size) < 64 or image.width * image.height > MAX_PIXELS:\n                raise ValueError(\'Фото: от 64 пикселей по стороне до 16 мегапикселей.\')\n            if getattr(image, \'n_frames\', 1) != 1:\n                raise ValueError(\'Нужно неподвижное фото.\')\n            return ImageOps.exif_transpose(image).convert(\'RGB\')\n    except (binascii.Error, OSError, Image.DecompressionBombError) as exc:\n        raise ValueError(\'Не удалось прочитать фото.\') from exc\n\n\nclass Session:\n    def __init__(self, pipe, max_generations=6):\n        self.pipe = pipe\n        self.maximum, self.attempts = max_generations, 0\n        self.lock = threading.Lock()\n        self.started = self.last_used = time.monotonic()\n\n    def status(self):\n        return dict(protocol=1, settings=SETTINGS, busy=self.lock.locked(),\n                    remaining=max(0, self.maximum-self.attempts), quality_verified=False)\n\n    def generate(self, payload):\n        import gradio as gr\n        if not isinstance(payload, dict) or set(payload) != {\'person\', \'garment\'}:\n            raise gr.Error(\'Нужны фото человека и одежды.\', print_exception=False)\n        try:\n            person, garment = decode_image(payload[\'person\']), decode_image(payload[\'garment\'])\n        except ValueError as exc:\n            raise gr.Error(str(exc), print_exception=False) from None\n        if not self.lock.acquire(blocking=False):\n            raise gr.Error(\'Уже создаётся кадр. Дождитесь его окончания.\', print_exception=False)\n        try:\n            if self.attempts >= self.maximum:\n                raise gr.Error(\'Лимит этой тестовой сессии исчерпан.\', print_exception=False)\n            self.attempts += 1\n            self.last_used = time.monotonic()\n            result = self.pipe(person_image=person, garment_image=garment,\n                category=\'tops\', garment_photo_type=\'flat-lay\', num_samples=1,\n                num_timesteps=30, guidance_scale=1.5, seed=42,\n                segmentation_free=True).images[0]\n            buf = io.BytesIO()\n            result.save(buf, format=\'PNG\')\n            return dict(image=\'data:image/png;base64,\' + base64.b64encode(buf.getvalue()).decode(),\n                        settings=SETTINGS, width=result.width, height=result.height,\n                        elapsed_seconds=round(time.monotonic()-self.last_used, 2))\n        except gr.Error:\n            raise\n        except Exception as exc:\n            print(\'Inference failed:\', type(exc).__name__, flush=True)\n            raise gr.Error(\'Генерация не завершилась. Проверьте журнал Kaggle; повторного запуска не было.\',\n                           print_exception=False) from None\n        finally:\n            self.last_used = time.monotonic()\n            self.lock.release()\n\n\ndef build_app(session):\n    import gradio as gr\n    with gr.Blocks(analytics_enabled=False) as app:\n        data = gr.JSON(visible=False)\n        output = gr.JSON(visible=False)\n        gr.Button(visible=False).click(session.generate, data, output,\n            api_name=\'tryon\', concurrency_limit=1, concurrency_id=\'gpu\')\n        gr.Button(visible=False).click(session.status, None, output,\n            api_name=\'status\', concurrency_limit=None, queue=False)\n    app.queue(max_size=2, default_concurrency_limit=1)\n    return app\n\n\ndef main():\n    from engine import download_weights, make_pipeline\n    import torch\n    from starlette.middleware import Middleware\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'Включите GPU в Kaggle. На CPU этот сервер не запускается.\')\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.backends.cudnn.allow_tf32 = False\n    root = Path(\'/kaggle/working/modegen_web\')\n    root.mkdir(parents=True, exist_ok=True)\n    print(\'Загружаю FASHN v1.1. Фото для запуска не требуются.\', flush=True)\n    pipe = make_pipeline(download_weights(root), \'cpu\', 512)\n    def finite_output(module, inputs, output):\n        if not torch.isfinite(output).all().item():\n            raise RuntimeError(\'Non-finite model output\')\n    pipe.tryon_model.final_layer.register_forward_hook(finite_output)\n    session = Session(pipe)\n    key = secrets.token_urlsafe(32)\n    app = build_app(session)\n    try:\n        _, _, share_url = app.launch(share=True, server_name=\'127.0.0.1\', server_port=7860,\n            prevent_thread_lock=True, show_error=False, quiet=True, max_threads=2,\n            app_kwargs={\'middleware\': [Middleware(AccessGuard, key=key)]})\n        if not share_url or not share_url.startswith(\'https://\'):\n            raise RuntimeError(\'Не удалось открыть HTTPS-подключение. Сервер остановлен.\')\n        print(\'\\nОткройте https://modegen.ru/ai-platform/ → Подключение Kaggle\', flush=True)\n        print(\'Адрес подключения:\', share_url, flush=True)\n        print(\'Ключ подключения:\', key, flush=True)\n        print(\'Ключ только для вашей вкладки сайта. Не публикуйте вывод notebook.\', flush=True)\n        print(\'Сессия: до 6 кадров; остановка через 30 минут простоя или 2 часа работы.\\n\'\n              \'Держите эту ячейку запущенной. По окончании остановите сессию Kaggle.\', flush=True)\n        while True:\n            time.sleep(5)\n            now = time.monotonic()\n            if not session.lock.locked() and (now-session.last_used > 1800 or\n                    now-session.started > 7200):\n                print(\'Тестовый сервер остановлен по времени. Остановите GPU-сессию Kaggle.\', flush=True)\n                break\n    finally:\n        app.close()\n\n\nif __name__ == \'__main__\':\n    main()\n'}
for name,source in SCRIPTS.items(): (ROOT/(name+'.py')).write_text(source)
subprocess.run([sys.executable,'-u',str(ROOT/'setup_runtime.py'),'--output',str(ROOT)],check=True)


In [ ]:
subprocess.run([sys.executable,'-u',str(ROOT/'server.py')],check=True)
